<a href="https://colab.research.google.com/github/Mohamad-101/Secure-Attendance-System-Using-Facial-Recognition/blob/main/research_sandbox/Phase1_Vision_and_DB_Tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1: Biometric Authentication & Relational Database Design
**Project:** Secure Attendance System using Facial Recognition
  
**Developer:** Mohamad El Saleh

**Host Institution:** International Center for AI and Cyber Security Research and Innovations

**Academic Affiliation:** Lebanese University Faculty of Engineering (ULFG)  

## Overview
This notebook serves as the Phase 1 proof-of-concept environment. It handles workspace initialization, facial embedding extraction, Euclidean distance threshold validation ($d \le 0.60$), and relational SQLite3 database mapping.

In [ ]:
# 1. Install core computer vision dependencies quietly
!pip install -q face_recognition opencv-python

import os

# 2. Create a dedicated folder for your testing assets
IMAGE_DIR = '/content/images'
os.makedirs(IMAGE_DIR, exist_ok=True)

print(f"Success: Environment ready. Target directory created at: {IMAGE_DIR}")

## Phase 1.1: Facial Embedding & Threshold Evaluation
This section loads the test image assets into memory, localizes facial targets, extracts their 128-dimensional vector spaces, and evaluates them against the system matching threshold.

In [ ]:
import os
import face_recognition

# Define generic asset paths
img1_path = "/content/images/baseline.jpeg"
img2_path = "/content/images/verification.jpeg"
stranger_path = "/content/images/imposter.jpeg"

# Safeguard check to verify image uploads
if not (os.path.exists(img1_path) and os.path.exists(img2_path) and os.path.exists(stranger_path)):
    raise FileNotFoundError(f"Missing files. Double check your filenames match exactly: {os.listdir('/content/images')}")

# Load images and extract 128-dimensional facial encodings
encoding_1 = face_recognition.face_encodings(face_recognition.load_image_file(img1_path))[0]
encoding_2 = face_recognition.face_encodings(face_recognition.load_image_file(img2_path))[0]
encoding_stranger = face_recognition.face_encodings(face_recognition.load_image_file(stranger_path))[0]

# Calculate absolute Euclidean distances
distance_match = face_recognition.face_distance([encoding_1], encoding_2)[0]
distance_stranger = face_recognition.face_distance([encoding_1], encoding_stranger)[0]

# Operational decision boundary
THRESHOLD = 0.60

print(f"Distance (Same User): {distance_match:.4f} | Authenticated? {distance_match <= THRESHOLD}")
print(f"Distance (Stranger) : {distance_stranger:.4f} | Authenticated? {distance_stranger <= THRESHOLD}")

## Phase 1.2: Relational Database Schema Design
This section configures the local SQLite3 architecture. Because relational database layers cannot natively store multi-dimensional NumPy arrays, the 128 floating-point vector values are flattened and serialized into standard text-based JSON strings.

In [ ]:
import sqlite3
import json
import os

db_path = "attendance_system.db"

# Automated sandbox cleanup: Wipe old database file if it exists to apply the new schema fresh
if os.path.exists(db_path):
    os.remove(db_path)
    print("[System Log]: Cleared legacy database file to apply fresh schema.")

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Enforce relational foreign key constraints
cursor.execute("PRAGMA foreign_keys = ON;")

# Create generalized enterprise schema tables
cursor.execute('''CREATE TABLE IF NOT EXISTS Users (
                    user_id TEXT PRIMARY KEY,
                    full_name TEXT NOT NULL)''')

cursor.execute('''CREATE TABLE IF NOT EXISTS Facial_Profiles (
                    user_id TEXT PRIMARY KEY,
                    face_encoding TEXT NOT NULL,
                    FOREIGN KEY (user_id) REFERENCES Users(user_id) ON DELETE CASCADE)''')

cursor.execute('''CREATE TABLE IF NOT EXISTS Attendance_Logs (
                    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    user_id TEXT NOT NULL,
                    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
                    date DATE DEFAULT (date('now', 'localtime')),
                    FOREIGN KEY (user_id) REFERENCES Users(user_id) ON DELETE CASCADE)''')

# Serialize multi-dimensional vector embedding to standard text-based JSON
serialized_vector = json.dumps(encoding_1.tolist())

# Operational testing placeholders (Generic production mock data)
MOCK_USER_ID = "USR-001"
MOCK_USER_NAME = "Default Test User"

# Execute transactions using clean, parameterized query syntax
cursor.execute("INSERT OR IGNORE INTO Users (user_id, full_name) VALUES (?, ?)",
               (MOCK_USER_ID, MOCK_USER_NAME))

cursor.execute("INSERT OR IGNORE INTO Facial_Profiles (user_id, face_encoding) VALUES (?, ?)",
               (MOCK_USER_ID, serialized_vector))
conn.commit()

# Execute schema validation query
cursor.execute("SELECT full_name FROM Users WHERE user_id = ?", (MOCK_USER_ID,))
user_record = cursor.fetchone()

conn.close()

print(f"\nDatabase Verified: Secure initialization pipeline verified for '{user_record[0]}'.")